<a href="https://colab.research.google.com/github/miserableFrog/fake-news-covid-19/blob/main/deep_learning_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

# Let's list the contents of your Drive to find the correct path
!ls "/content/drive/My Drive"

# Prompt the user for the correct file path
file_path = input("Enter the correct path to your CSV file (including the filename): ")

# Ensure the file path starts from 'My Drive' to avoid issues
if not file_path.startswith('/content/drive/My Drive/'):
    file_path = '/content/drive/My Drive/' + file_path  # Prepend the necessary part

df = pd.read_csv(file_path)
print(df.head()) # Display the first few rows to verify

In [ ]:
!pip install tensorflow keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 1.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from keras_tuner import RandomSearch

In [ ]:
# Step 4: Preprocess the data

data = df
text_data = data['text_nostop'].fillna('')
target = data['target'].apply(lambda x: 1 if x.lower() == 'true' else 0)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(text_data, target, test_size=0.2, random_state=42)

# Tokenize and pad sequences
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

max_len = 100  # Set max length for padding
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_len, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_len, padding='post', truncating='post')

In [ ]:
# Step 5: Define a function to build the Bidirectional LSTM model

def build_model(hp):
    model = Sequential()
    model.add(Embedding(input_dim=5000, output_dim=hp.Int('embedding_dim', min_value=32, max_value=128, step=32), input_length=max_len))
    model.add(Bidirectional(LSTM(hp.Int('lstm_units', min_value=32, max_value=128, step=32), return_sequences=False)))
    model.add(Dropout(hp.Float('dropout', min_value=0.2, max_value=0.5, step=0.1)))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=tf.keras.optimizers.Adam(hp.Choice('learning_rate', values=[1e-3, 5e-4])),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model

In [ ]:
# Step 6: Hyperparameter Tuning with Keras Tuner
tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,  # Number of combinations to try
    executions_per_trial=1,  # Number of models to train per trial
    directory='my_dir',
    project_name='lstm_tuning'
)

tuner.search(X_train_padded, np.array(y_train), epochs=3, validation_split=0.2)

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best hyperparameters: {best_hps}")

Trial 5 Complete [00h 00m 57s]
val_accuracy: 0.8235294222831726

Best val_accuracy So Far: 0.8470588326454163
Total elapsed time: 00h 02m 24s
Best hyperparameters: <keras_tuner.src.engine.hyperparameters.hyperparameters.HyperParameters object at 0x783ffb84ffa0>


In [ ]:
# Step 7: Build and Train the Final Model
model = tuner.hypermodel.build(best_hps)
model.fit(X_train_padded, np.array(y_train), epochs=5, validation_split=0.2, batch_size=32)

# Step 8: Evaluate the Model
loss, accuracy = model.evaluate(X_test_padded, np.array(y_test))
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 119ms/step - accuracy: 0.7411 - loss: 0.6355 - val_accuracy: 0.7451 - val_loss: 0.5627
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 93ms/step - accuracy: 0.7806 - loss: 0.4853 - val_accuracy: 0.7451 - val_loss: 0.4817
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 111ms/step - accuracy: 0.8847 - loss: 0.2635 - val_accuracy: 0.8235 - val_loss: 0.3836
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 97ms/step - accuracy: 0.9936 - loss: 0.1013 - val_accuracy: 0.8118 - val_loss: 0.5315
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 93ms/step - accuracy: 0.9986 - loss: 0.0346 - val_accuracy: 0.8314 - val_loss: 0.4764
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8538 - loss: 0.4519
Test Accuracy: 85.22%


In [ ]:
# prompt: get all values from best_hps

# Get all values from best_hps
for hp_name, hp_value in best_hps.values.items():
  print(f"{hp_name}: {hp_value}")


embedding_dim: 128
lstm_units: 32
dropout: 0.4
learning_rate: 0.001
